# TI 2026 — Main Event Game & Fantasy Summary

Companion to the group-stage analysis. This notebook summarizes playoff games, series results, team form, role/player fantasy output, consistency, and the stats most associated with a strong fantasy score.

> Run all cells from either the repository root or the `notebooks/` directory. All tables and charts are generated from `data/2026/main_event`.

## 0. Setup and data loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')
ROLE_ORDER = ['core', 'mid', 'support']
ROLE_COLORS = {'core': '#d95f5f', 'mid': '#6f63c2', 'support': '#2a9d8f'}

In [ ]:
def find_data_dir():
    candidates = [Path('data/2026/main_event'), Path('../data/2026/main_event')]
    for candidate in candidates:
        if (candidate / 'fantasy_score.csv').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find data/2026/main_event. Start Jupyter from the repo root or notebooks directory.')

DATA_DIR = find_data_dir()
games = pd.read_csv(DATA_DIR / 'total.csv')
fantasy = pd.read_csv(DATA_DIR / 'fantasy_score.csv')
series_scores = pd.read_csv(DATA_DIR / 'series_scores.csv')
series_stats = pd.read_csv(DATA_DIR / 'series_stat_scores.csv')

for frame in (games, fantasy, series_scores, series_stats):
    frame['teamName'] = frame['teamName'].astype(str).str.strip()
    if 'Versus' in frame.columns:
        frame['Versus'] = frame['Versus'].astype(str).str.strip()

for frame in (games, fantasy):
    frame['date'] = pd.to_datetime(frame['date'], dayfirst=True, errors='coerce')

required = {'matchID', 'seriesID', 'teamName', 'position', 'playerName'}
missing = required - set(fantasy.columns)
assert not missing, f'Missing expected fantasy columns: {sorted(missing)}'

STAT_COLUMNS = [
    'kills', 'deaths', 'creep_score', 'gpm', 'madstone_collected', 'tower_kills',
    'obs_placed', 'camps_stacked', 'runes_grabbed', 'watchers_taken', 'smokes_used',
    'teamfight_participation', 'stuns', 'firstblood_claimed', 'roshan_kills',
    'tormentor_kills', 'courier_kills'
]
STAT_COLUMNS = [column for column in STAT_COLUMNS if column in fantasy.columns]
fantasy[STAT_COLUMNS] = fantasy[STAT_COLUMNS].fillna(0)
fantasy['fantasy_total'] = fantasy[STAT_COLUMNS].sum(axis=1)

print(f'Data directory: {DATA_DIR}')
print(f'{games.matchID.nunique()} games | {games.seriesID.nunique()} series IDs | {games.teamName.nunique()} teams | {games.playerID.nunique()} players')
print(f'Dates: {games.date.min():%d %b %Y} — {games.date.max():%d %b %Y}')

## 1. Tournament overview

In [ ]:
# One row per team per game. `win` is repeated on the player rows, so `first` is sufficient.
team_games = (
    games.groupby(['matchID', 'seriesID', 'date', 'Versus', 'teamID', 'teamName'], as_index=False)
         .agg(win=('win', 'first'), gametime=('gametime', 'first'), game_number=('Game_number', 'first'))
)
team_games['win'] = team_games['win'].astype(int)
team_games['duration_min'] = team_games['gametime'] / 60

team_summary = (
    team_games.groupby('teamName', as_index=False)
              .agg(games=('matchID', 'nunique'), wins=('win', 'sum'),
                   avg_duration_min=('duration_min', 'mean'))
)
team_summary['losses'] = team_summary['games'] - team_summary['wins']
team_summary['win_rate'] = team_summary['wins'] / team_summary['games']
team_summary = team_summary.sort_values(['win_rate', 'wins'], ascending=False)
display(team_summary.style.format({'win_rate': '{:.1%}', 'avg_duration_min': '{:.1f}'}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=team_summary, y='teamName', x='win_rate', color='#457b9d', ax=axes[0])
axes[0].set(title='Main-event game win rate', xlabel='Win rate', ylabel='')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f'{value:.0%}'))
sns.barplot(data=team_summary, y='teamName', x='games', color='#e9c46a', ax=axes[1])
axes[1].set(title='Games played', xlabel='Games', ylabel='')
plt.tight_layout()
plt.show()

In [ ]:
# Compact chronological game log.
match_log = (
    team_games.assign(result=np.where(team_games['win'].eq(1), 'W', 'L'))
              .sort_values(['date', 'matchID', 'teamName'])
              [['date', 'seriesID', 'matchID', 'game_number', 'teamName', 'Versus', 'result', 'duration_min']]
)
display(match_log.style.format({'date': lambda value: value.strftime('%d %b') if pd.notna(value) else '',
                                'duration_min': '{:.1f}'}))

## 2. Series summary

The data includes a few `single_...` series IDs. They are retained so the notebook faithfully summarizes every supplied main-event game.

In [ ]:
series_results = (
    team_games.groupby(['seriesID', 'Versus', 'teamName'], as_index=False)
              .agg(games=('matchID', 'nunique'), wins=('win', 'sum'), first_date=('date', 'min'))
)
series_results['losses'] = series_results['games'] - series_results['wins']
series_results['record'] = series_results['wins'].astype(str) + '–' + series_results['losses'].astype(str)
series_results['won_series'] = series_results['wins'] > series_results['losses']
series_results = series_results.sort_values(['first_date', 'seriesID', 'wins'], ascending=[True, True, False])
display(series_results[['first_date', 'seriesID', 'Versus', 'teamName', 'record', 'won_series']].style
        .format({'first_date': lambda value: value.strftime('%d %b') if pd.notna(value) else ''}))

series_record = (
    series_results.groupby('teamName', as_index=False)
                  .agg(series_played=('seriesID', 'nunique'), series_wins=('won_series', 'sum'))
)
series_record['series_win_rate'] = series_record['series_wins'] / series_record['series_played']
display(series_record.sort_values('series_win_rate', ascending=False).style.format({'series_win_rate': '{:.1%}'}))

## 3. Game length and pace

In [ ]:
unique_matches = team_games.drop_duplicates('matchID').copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(unique_matches['duration_min'], bins='auto', kde=True, color='#2a9d8f', ax=axes[0])
axes[0].axvline(unique_matches['duration_min'].median(), color='black', linestyle='--',
                label=f"Median: {unique_matches['duration_min'].median():.1f} min")
axes[0].legend()
axes[0].set(title='Game-duration distribution', xlabel='Minutes', ylabel='Games')
duration_by_team = team_games.groupby('teamName', as_index=False)['duration_min'].mean().sort_values('duration_min')
sns.barplot(data=duration_by_team, y='teamName', x='duration_min', color='#f4a261', ax=axes[1])
axes[1].set(title='Average game duration by team', xlabel='Minutes', ylabel='')
plt.tight_layout()
plt.show()
display(unique_matches.nlargest(5, 'duration_min')[['date', 'matchID', 'Versus', 'duration_min']]
        .style.format({'date': lambda value: value.strftime('%d %b') if pd.notna(value) else '', 'duration_min': '{:.1f}'}))

## 4. Fantasy scoring by team and role

`series_scores.csv` is the authoritative top-two-games series total produced by the project pipeline.

In [ ]:
role_series = (
    series_scores.groupby(['teamName', 'position'], as_index=False)
                 .agg(series=('seriesID', 'nunique'), avg_series_score=('series_score', 'mean'),
                      best_series_score=('series_score', 'max'), score_sd=('series_score', 'std'))
)
role_series['position'] = pd.Categorical(role_series['position'], ROLE_ORDER, ordered=True)
role_series = role_series.sort_values(['position', 'avg_series_score'], ascending=[True, False])
display(role_series.style.format({'avg_series_score': '{:,.0f}', 'best_series_score': '{:,.0f}', 'score_sd': '{:,.0f}'}))

plt.figure(figsize=(13, 7))
sns.barplot(data=role_series, y='teamName', x='avg_series_score', hue='position',
            hue_order=ROLE_ORDER, palette=ROLE_COLORS)
plt.title('Average top-two-games fantasy score per series')
plt.xlabel('Average series score')
plt.ylabel('')
plt.legend(title='Role')
plt.tight_layout()
plt.show()

## 5. Player leaderboard and consistency

Average and ceiling are useful for upside; median and coefficient of variation help identify safer picks. Lower CV means more consistent scoring.

In [ ]:
player_summary = (
    fantasy.groupby(['playerID', 'playerName', 'teamName', 'position'], as_index=False)
           .agg(games=('matchID', 'nunique'), avg_score=('fantasy_total', 'mean'),
                median_score=('fantasy_total', 'median'), best_game=('fantasy_total', 'max'),
                score_sd=('fantasy_total', 'std'))
)
player_summary['cv'] = player_summary['score_sd'] / player_summary['avg_score'].replace(0, np.nan)
player_summary['position'] = pd.Categorical(player_summary['position'], ROLE_ORDER, ordered=True)
eligible_players = player_summary[player_summary['games'] >= 2].copy()

for role in ROLE_ORDER:
    print(f'\nTop {role.title()} players by average score')
    display(eligible_players[eligible_players['position'] == role]
            .nlargest(10, 'avg_score')
            [['playerName', 'teamName', 'games', 'avg_score', 'median_score', 'best_game', 'cv']]
            .style.format({'avg_score': '{:,.0f}', 'median_score': '{:,.0f}',
                           'best_game': '{:,.0f}', 'cv': '{:.2f}'}))

In [ ]:
plot_players = (eligible_players.sort_values('avg_score', ascending=False)
                .groupby('position', observed=True).head(8))
g = sns.catplot(data=plot_players, y='playerName', x='avg_score', col='position',
                col_order=ROLE_ORDER, kind='bar', sharex=False, sharey=False,
                color='#457b9d', height=5, aspect=0.85)
g.set_axis_labels('Average fantasy score per game', '')
g.set_titles('{col_name}')
g.figure.suptitle('Top fantasy performers by role', y=1.04, fontsize=16)
plt.show()

safe_picks = (eligible_players[eligible_players['avg_score'] >= eligible_players.groupby('position', observed=True)['avg_score'].transform('median')]
              .sort_values(['position', 'cv']))
display(safe_picks[['position', 'playerName', 'teamName', 'games', 'avg_score', 'cv']].head(20)
        .style.format({'avg_score': '{:,.0f}', 'cv': '{:.2f}'}))

## 6. Which stats matter most?

The heatmap shows Spearman correlation between each scored component and the per-game fantasy total within each role. It is descriptive, not causal, and naturally favors large/high-variance scoring components.

In [ ]:
correlations = {}
for role in ROLE_ORDER:
    role_data = fantasy[fantasy['position'] == role]
    correlations[role] = role_data[STAT_COLUMNS + ['fantasy_total']].corr(method='spearman')['fantasy_total'].drop('fantasy_total')
corr_table = pd.DataFrame(correlations).sort_values('core', ascending=False)

plt.figure(figsize=(9, 10))
sns.heatmap(corr_table, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1, linewidths=.5)
plt.title('Fantasy-total correlation by role')
plt.xlabel('Role')
plt.ylabel('Scoring component')
plt.tight_layout()
plt.show()
display(corr_table.style.format('{:.2f}').background_gradient(cmap='vlag', vmin=-1, vmax=1))

## 7. Reusable stat deep dive

This compact helper replaces the repeated per-stat cells in the original notebook. It reports the best games, the distribution, and team averages for any role/stat combination.

In [ ]:
def analyze_stat(role, stat, top_n=10):
    if role not in ROLE_ORDER:
        raise ValueError(f'role must be one of {ROLE_ORDER}')
    if stat not in STAT_COLUMNS:
        raise ValueError(f'Unknown stat. Choose one of: {STAT_COLUMNS}')

    role_data = fantasy[fantasy['position'] == role].copy()
    leaders = role_data.nlargest(top_n, stat)[['date', 'Versus', 'playerName', 'teamName', stat]]
    display(leaders.style.format({'date': lambda value: value.strftime('%d %b') if pd.notna(value) else '', stat: '{:,.2f}'}))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.histplot(role_data[stat], kde=True, color=ROLE_COLORS[role], ax=axes[0])
    axes[0].set(title=f'{role.title()} {stat.replace("_", " ")} distribution', xlabel='Points per player-game')
    team_stat = role_data.groupby('teamName', as_index=False)[stat].mean().sort_values(stat, ascending=False)
    sns.barplot(data=team_stat, y='teamName', x=stat, color=ROLE_COLORS[role], ax=axes[1])
    axes[1].set(title='Average by team', xlabel='Points per player-game', ylabel='')
    plt.tight_layout()
    plt.show()

# Examples — change these arguments to explore other role/stat combinations.
analyze_stat('core', 'creep_score')
analyze_stat('mid', 'teamfight_participation')
analyze_stat('support', 'camps_stacked')

## 8. Best players when building a fantasy team

Main-event cards use **five stat slots per role**. Core and support teammates are averaged into one role score, mid is scored solo, and the best two game scores form the series score. Configure the five stats below to match your cards; repeated stats and multipliers are supported.

In [ ]:
# Suggested balanced setup based on the group-stage findings.
# Each tuple is (stat, multiplier). Change these to match your actual card slots.
MAIN_EVENT_BUILD = {
    'core': [
        ('creep_score', 1.0), ('teamfight_participation', 1.0),
        ('gpm', 1.0), ('stuns', 1.0), ('deaths', 1.0),
    ],
    'mid': [
        ('creep_score', 1.0), ('runes_grabbed', 1.0),
        ('teamfight_participation', 1.0), ('gpm', 1.0), ('stuns', 1.0),
    ],
    'support': [
        ('camps_stacked', 1.0), ('teamfight_participation', 1.0),
        ('obs_placed', 1.0), ('stuns', 1.0), ('smokes_used', 1.0),
    ],
}

# Set to a list such as ['Team Spirit', 'Team Liquid'] for the next round.
# None considers every team in the supplied main-event data.
ELIGIBLE_TEAMS = None
MIN_SERIES = 1

for role, slots in MAIN_EVENT_BUILD.items():
    invalid = [stat for stat, _ in slots if stat not in STAT_COLUMNS]
    assert len(slots) == 5, f'{role} must contain exactly five slots'
    assert not invalid, f'{role} contains unavailable stats: {invalid}'

fantasy_build = fantasy.copy()
fantasy_build['selected_score'] = 0.0
for role, slots in MAIN_EVENT_BUILD.items():
    role_mask = fantasy_build['position'].eq(role)
    fantasy_build.loc[role_mask, 'selected_score'] = sum(
        fantasy_build.loc[role_mask, stat] * multiplier for stat, multiplier in slots
    )

# Average the two core/support players within a game; mid naturally remains solo.
role_games = (
    fantasy_build.groupby(
        ['seriesID', 'Versus', 'teamID', 'teamName', 'position', 'matchID'], as_index=False
    ).agg(game_score=('selected_score', 'mean'))
)

# Apply the real series rule: sum the best two combined game scores.
series_role_scores = (
    role_games.groupby(['seriesID', 'Versus', 'teamID', 'teamName', 'position'], as_index=False)
              .agg(games=('matchID', 'nunique'),
                   series_score=('game_score', lambda values: values.nlargest(2).sum()))
)

role_players = (
    fantasy.groupby(['teamID', 'teamName', 'position'])['playerName']
           .agg(lambda values: ' & '.join(pd.unique(values.astype(str))))
           .rename('players').reset_index()
)
candidate_scores = (
    series_role_scores.groupby(['teamID', 'teamName', 'position'], as_index=False)
                      .agg(series=('seriesID', 'nunique'), expected=('series_score', 'mean'),
                           floor=('series_score', 'median'), upside=('series_score', 'max'),
                           volatility=('series_score', 'std'))
                      .merge(role_players, on=['teamID', 'teamName', 'position'], how='left')
)
candidate_scores['volatility'] = candidate_scores['volatility'].fillna(0)
candidate_scores['safe_score'] = candidate_scores['expected'] - 0.5 * candidate_scores['volatility']
candidate_scores = candidate_scores[candidate_scores['series'] >= MIN_SERIES].copy()
if ELIGIBLE_TEAMS is not None:
    candidate_scores = candidate_scores[candidate_scores['teamName'].isin(ELIGIBLE_TEAMS)].copy()
candidate_scores['position'] = pd.Categorical(candidate_scores['position'], ROLE_ORDER, ordered=True)

for role in ROLE_ORDER:
    print(f'\nBest {role.title()} choice for the configured card')
    columns = ['players', 'teamName', 'series', 'expected', 'floor', 'upside', 'volatility', 'safe_score']
    display(candidate_scores[candidate_scores['position'].eq(role)]
            .sort_values(['expected', 'safe_score'], ascending=False)[columns].head(10)
            .style.format({column: '{:,.0f}' for column in ['expected', 'floor', 'upside', 'volatility', 'safe_score']}))

In [ ]:
from itertools import product

def best_fantasy_lineups(candidates, metric='expected', max_roles_per_team=None, top_n=10):
    """Combine one core, one mid, and one support candidate into ranked lineups.

    metric: 'expected' for maximum average, 'safe_score' for consistency, or 'upside' for ceiling.
    max_roles_per_team: set to 1 or 2 if your rules limit cards from the same team.
    """
    if metric not in {'expected', 'safe_score', 'upside'}:
        raise ValueError("metric must be 'expected', 'safe_score', or 'upside'")

    pools = [candidates[candidates['position'].eq(role)].to_dict('records') for role in ROLE_ORDER]
    if any(not pool for pool in pools):
        raise ValueError('At least one role has no eligible candidates. Check ELIGIBLE_TEAMS and MIN_SERIES.')

    lineups = []
    for core, mid, support in product(*pools):
        selections = [core, mid, support]
        team_counts = pd.Series([pick['teamName'] for pick in selections]).value_counts()
        if max_roles_per_team is not None and team_counts.max() > max_roles_per_team:
            continue
        lineups.append({
            'core': f"{core['players']} ({core['teamName']})",
            'mid': f"{mid['players']} ({mid['teamName']})",
            'support': f"{support['players']} ({support['teamName']})",
            'lineup_score': sum(pick[metric] for pick in selections),
        })
    return pd.DataFrame(lineups).nlargest(top_n, 'lineup_score').reset_index(drop=True)

print('Highest expected-score lineups')
display(best_fantasy_lineups(candidate_scores, metric='expected')
        .style.format({'lineup_score': '{:,.0f}'}))

print('Safer lineups (average minus half a standard deviation)')
display(best_fantasy_lineups(candidate_scores, metric='safe_score')
        .style.format({'lineup_score': '{:,.0f}'}))

print('Highest-ceiling lineups')
display(best_fantasy_lineups(candidate_scores, metric='upside')
        .style.format({'lineup_score': '{:,.0f}'}))

## 9. Reading the results

After running all cells, use these checks when selecting fantasy players:

- **Expected series length:** more games create more chances for a strong top-two total.
- **Floor vs ceiling:** average/median and CV indicate safety; best game indicates upside.
- **Role-specific drivers:** use the correlation heatmap and deep dive rather than applying core priorities to supports.
- **Small samples:** main-event elimination means some teams and players have very few games; always read `games` beside the score.
- **Context:** descriptive tournament results do not account for opponent strength, hero pool, patch shifts, or bracket position.